# 04 - Backtesting: Bollinger Bands Mean Reversion

**Capítulo**: 04 - Mean Reversion

**Objetivo**: Backtest con stop dinámico (ATR), comparación salida en banda media vs superior.

---

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
from backtesting import Backtest, Strategy

from curso.lib.data import download_historical
from curso.lib.backtest import run_backtest, extract_metrics, metrics_to_dataframe, compare_strategies
from curso.lib.reporting import plot_equity_curve, plot_drawdown, print_metrics_table, plot_comparison

import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
print('Setup completado ✓')

## 1. Datos

In [ ]:
TICKER = 'XLF'
df = download_historical(TICKER)
print(f'{TICKER}: {len(df)} registros')

## 2. Estrategia Bollinger

In [ ]:
def BB_upper(values, n, k):
    s = pd.Series(values)
    return (s.rolling(n).mean() + k * s.rolling(n).std()).values

def BB_lower(values, n, k):
    s = pd.Series(values)
    return (s.rolling(n).mean() - k * s.rolling(n).std()).values

def BB_middle(values, n):
    return pd.Series(values).rolling(n).mean().values

def ATR_calc(high, low, close, n):
    h, l, c = pd.Series(high), pd.Series(low), pd.Series(close)
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    return tr.rolling(n).mean().values


class BollingerMeanReversion(Strategy):
    bb_period = 20
    bb_std = 2.0
    atr_period = 14
    atr_mult = 2.0
    
    def init(self):
        self.bb_low = self.I(BB_lower, self.data.Close, self.bb_period, self.bb_std)
        self.bb_mid = self.I(BB_middle, self.data.Close, self.bb_period)
        self.bb_up = self.I(BB_upper, self.data.Close, self.bb_period, self.bb_std)
        self.atr = self.I(ATR_calc, self.data.High, self.data.Low, self.data.Close, self.atr_period)
    
    def next(self):
        price = self.data.Close[-1]
        
        if not self.position:
            if price < self.bb_low[-1]:
                sl = price - self.atr_mult * self.atr[-1]
                self.buy(sl=sl)
        else:
            if price >= self.bb_mid[-1]:
                self.position.close()

## 3. Backtest

In [ ]:
stats, bt = run_backtest(df, BollingerMeanReversion)
metrics = extract_metrics(stats)
print_metrics_table(metrics_to_dataframe(metrics))

fig = plot_equity_curve(stats, title=f'{TICKER} - Bollinger Mean Reversion')
plt.show()

## 4. Variante: salida en banda superior

In [ ]:
class BollingerUpperExit(BollingerMeanReversion):
    def next(self):
        price = self.data.Close[-1]
        if not self.position:
            if price < self.bb_low[-1]:
                sl = price - self.atr_mult * self.atr[-1]
                self.buy(sl=sl)
        else:
            if price >= self.bb_up[-1]:
                self.position.close()

stats_up, _ = run_backtest(df, BollingerUpperExit)
comparison = compare_strategies({'Exit BB Media': stats, 'Exit BB Superior': stats_up})
plot_comparison(comparison)
plt.show()

## 5. Conclusiones

In [ ]:
print('''
CONCLUSIONES
============
1. Mean reversion: [¿funciona en este activo?]
2. Exit target: [¿mejor salir en media o banda superior?]
3. Stop ATR: [¿cuántas salidas por stop vs señal?]
4. DECISIÓN: [aprobar / iterar]
''')